Logsitic Regression and Feature Scaling

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split

I have one of three datasets that is designed to predict a binary yes or no, so that is the only dataset I will work with this week. The variable "DEP_DEL15" is 1 is a flight was delayed greater than 15 minutes at departure and 0 if it was not.

In [2]:
#Import 2nd Dataset, 2019 Flight Delay Info

delays_2019 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2019_Delay_Data\full_data_flightdelay.csv")

delays_2019.head()

,MONTH,DAY_OF_WEEK,DEP_DEL15,DEP_TIME_BLK,DISTANCE_GROUP,SEGMENT_NUMBER,CONCURRENT_FLIGHTS,NUMBER_OF_SEATS,CARRIER_NAME,AIRPORT_FLIGHTS_MONTH,...,PLANE_AGE,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,PRCP,SNOW,SNWD,TMAX,AWND
0,1,7,0,0800-0859,2,1,25,143,Southwest Airlines Co.,13056,...,8,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
1,1,7,0,0700-0759,7,1,29,191,Delta Air Lines Inc.,13056,...,3,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
2,1,7,0,0600-0659,7,1,27,199,Delta Air Lines Inc.,13056,...,18,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
3,1,7,0,0600-0659,9,1,27,180,Delta Air Lines Inc.,13056,...,2,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
4,1,7,0,0001-0559,7,1,10,182,Spirit Air Lines,13056,...,1,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91


In [3]:
#The target variable for this dataset is "DEP_DEL15" it is a binary 1 if the flight was delayed more than 15 minutes in its departure delay else 0
num_delays_2019 = delays_2019.drop(["DEP_TIME_BLK", "CARRIER_NAME", "DEPARTING_AIRPORT", "PREVIOUS_AIRPORT"], axis = 1)

target_2019 = num_delays_2019["DEP_DEL15"]
variables_2019 = num_delays_2019.drop(["DEP_DEL15"], axis = 1)

I am going to use Standardization to scale the dataset since there are outliers. Normalization would have the values compressed by those outliers.

In [4]:
#Scale features
scaler = StandardScaler()
variables_scl_2019 = scaler.fit_transform(variables_2019)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl_2019, target_2019, random_state=42)

I am going to use Precision as the scoring metric since this is predicting significantly delayed flights. Most flights are not delayed so I want to minimize false positives.

In [10]:
#Optimize Logistic Regression
l1_ratio = np.linspace(0,1,5)
log_cv = LogisticRegressionCV(Cs=5, l1_ratios=l1_ratio, solver = 'saga', scoring="precision", random_state=42).fit(X_train, y_train)
print(f"Best performing C (inverse of regularization strength) = {log_cv.C_[0]:.5f}")
print(f"Best performing l1_ratio = {log_cv.l1_ratio_[0]:.5f}")

#Utilize Best Alpha
log_model = LogisticRegression(C=log_cv.C_[0],l1_ratio=log_cv.l1_ratio_[0], solver="saga").fit(X_train,y_train)
train_preds = log_model.predict(X_train)
test_preds = log_model.predict(X_test)
train_prec = metrics.precision_score(y_train,train_preds)
test_prec = metrics.precision_score(y_test,test_preds)

print(f"The precision on the training data was {train_prec:.4f} and on the test data was {test_prec:.4f}")

C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


Best performing C (inverse of regularization strength) = 0.00010
Best performing l1_ratio = 0.75000
The precision on the training data was 0.4989 and on the test data was 0.5080


In [13]:
print("Variables\t\t\t\tWeights")
print("-"*50)

for v,c in zip(variables_2019.columns, log_model.coef_[0]):
    print(f"{v:<25}\t\t{c:.6f}")

Variables				Weights
--------------------------------------------------
MONTH                    		-0.028565
DAY_OF_WEEK              		0.000000
DISTANCE_GROUP           		0.103187
SEGMENT_NUMBER           		0.329570
CONCURRENT_FLIGHTS       		-0.008627
NUMBER_OF_SEATS          		0.076213
AIRPORT_FLIGHTS_MONTH    		0.055907
AIRLINE_FLIGHTS_MONTH    		0.000000
AIRLINE_AIRPORT_FLIGHTS_MONTH		-0.012387
AVG_MONTHLY_PASS_AIRPORT 		0.000000
AVG_MONTHLY_PASS_AIRLINE 		-0.004718
FLT_ATTENDANTS_PER_PASS  		0.000000
GROUND_SERV_PER_PASS     		-0.019188
PLANE_AGE                		0.001907
LATITUDE                 		0.000000
LONGITUDE                		0.087736
PRCP                     		0.152172
SNOW                     		0.068952
SNWD                     		0.035280
TMAX                     		0.010497
AWND                     		0.093950


The precision of the model is underwhelming with only half of the predicted significant delays being actually significantly delayed flights. Segment number (# of the flight for that plane that day) and precipitation level are most relied on variables for the model. 

From what I know of this dataset from previous classes, segment number first increases with delay frequency before dropping off quite fast. I suspect then a lot of the false positives are the flights with the highest segment numbers which are actually not that frequently delayed.